In [ ]:
!pip install langchain langchain_huggingface langchain_community huggingface_hub
!pip install chromadb
!pip install accelerate
!huggingface-cli login --token "apitoken"

In [ ]:
!pip install langchain-chroma

In [ ]:
import os
from typing import Dict, Any, List, Optional, Tuple, Union
import torch
import shutil

# LangChain imports
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.embeddings import Embeddings
from langchain.prompts import PromptTemplate
from langchain_core.language_models import BaseLLM

from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_chroma import Chroma
from langchain.chains import RetrievalQA

from pathlib import Path

__file__ = "/kaggle/working/"

In [ ]:
# Source (read-only)
src_dir = '/kaggle/input/uni-vector-21-7-2025'
# Destination (writable)
dst_dir = '/kaggle/working/uni-vector-21-7-2025'

# Copy to a writable location if it doesn't already exist
if not os.path.exists(dst_dir):
    shutil.copytree(src_dir, dst_dir)
from langchain_chroma import Chroma
EMBEDDING_NAME = "intfloat/multilingual-e5-small"
embedding = HuggingFaceEmbeddings(model_name=EMBEDDING_NAME)
vectordb = Chroma(persist_directory='/kaggle/working/uni-vector-21-7-2025', embedding_function=embedding)
retriever = vectordb.as_retriever(search_kwargs={"k": 2})

In [ ]:
# Base directories
class config:
    """Configuration for the RAG system."""
    BASE_DIR = Path(os.path.dirname(os.path.abspath(__file__))).parent
    VECTOR_DB_DIR = '/kaggle/working/uni-vector-21-7-2025'

    # Embedding configuration
    EMBEDDING_MODEL = "intfloat/multilingual-e5-small"  # Vietnamese/multilingual support
    EMBEDDING_CACHE = os.path.join(BASE_DIR, "cache", "embedding_cache")

    # Retriever configuration
    DEFAULT_TOP_K = 5
    SIMILARITY_THRESHOLD = 0.7
    DEFAULT_TEMPERATURE = 0.3

    DEFAULT_LLM_MODEL = "meta-llama/Llama-3.2-1B" 
    HUGGINGFACE_LLM_MODEL = "meta-llama/Llama-3.2-1B"
    
    DEFAULT_RAG_PROMPT_TEMPLATE = """
Bạn là một AI tư vấn tuyển sinh đại học, dựa vào hiểu biết của mình, có thể sử dụng kèm theo những thông tin sau đây, hãy trả lời câu hỏi hoặc đưa ra tư vấn.

Thông tin được cung cấp:
{context}

Câu hỏi:
{question}

Trả lời:
"""

def get_default_embeddings() -> Embeddings:
    os.makedirs(config.EMBEDDING_CACHE, exist_ok=True)
    
    return HuggingFaceEmbeddings(
        model_name=config.EMBEDDING_MODEL,
        cache_folder=config.EMBEDDING_CACHE,
        model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"}
    )

def create_huggingface_llm(
    model_name: str = "meta-llama/Llama-3.2-1B",
    temperature: float = 0.7,
    max_tokens: Optional[int] = None,
    **kwargs
) -> BaseLLM:
    try:
        print(f"Loading HuggingFace model: {model_name}")
        # Initialize tokenizer
        tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True
        )
        
        # Add pad token if it doesn't exist
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        # For Llama and Qwen models, load the model directly
        if "llama" in model_name.lower() or "qwen" in model_name.lower():
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                trust_remote_code=True,
                device_map="auto",
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
            )
            
            pipe_params = {
                "model": model,
                "tokenizer": tokenizer,
                "do_sample": True,
                "temperature": temperature,
                "top_p": 0.95,
                "repetition_penalty": 1.15,
                **kwargs
            }
        else:
            pipe_params = {
                "tokenizer": tokenizer,
                "model": model_name,
                "device": kwargs.get("device", 0 if torch.cuda.is_available() else -1),
                "do_sample": True,
                "temperature": temperature,
                "pad_token_id": tokenizer.eos_token_id,
                **kwargs
            }
        
        # Set max_length if provided
        if max_tokens is not None:
            pipe_params["max_length"] = max_tokens
        
        # Create HuggingFace pipeline
        pipe = pipeline(
            "text-generation",
            **pipe_params
        )
        
        # Create LangChain LLM
        llm = HuggingFacePipeline(pipeline=pipe)
        print("HuggingFace model initialized successfully!")
        return llm
        
    except Exception as e:
        print(f"Error initializing HuggingFace model: {e}")
        return None

In [ ]:
class SimplifiedRAG:
    def __init__(
        self, 
        vector_db_path: str = config.VECTOR_DB_DIR,
        embedding_model: str = config.EMBEDDING_MODEL,
        llm_type: str = "huggingface",
        llm_model_name: str = None,
        temperature: float = config.DEFAULT_TEMPERATURE,
        top_k: int = config.DEFAULT_TOP_K
    ):
        if vector_db_path is None:
            vector_db_path = config.VECTOR_DB_DIR
        
        if not os.path.exists(vector_db_path):
            raise ValueError(f"Vector database not found at {vector_db_path}")
        
        self.embeddings = HuggingFaceEmbeddings(
            model_name=embedding_model,
            cache_folder="./cache/embedding_cache"
        )
        
        self.vectorstore = Chroma(
            persist_directory=vector_db_path,
            embedding_function=self.embeddings
        )
        
        self.retriever = self.vectorstore.as_retriever(
            search_kwargs={"k": top_k}
        )
        
        llm_model_name = config.HUGGINGFACE_LLM_MODEL

        self.llm = create_huggingface_llm(
            model_name=llm_model_name,
            temperature=temperature
        )
        
        self.temperature = temperature
        self.top_k = top_k
            
        # Create RAG chain
        prompt_template = PromptTemplate(
            template=config.DEFAULT_RAG_PROMPT_TEMPLATE,
            input_variables=["context", "question"]
        )

        self.qa_chain_huggingface = RetrievalQA.from_chain_type(
            llm=self.llm,
            chain_type="stuff",
            retriever=self.retriever,
            chain_type_kwargs={"prompt": prompt_template},
            return_source_documents=True
        )
        
    def ask(self, question: str) -> Dict[str, Any]:
        try:
            result = self.qa_chain_huggingface({"query": question})
            return {
                "question": question,
                "answer": result["result"],
                "source_documents": result["source_documents"]
            }
        except Exception as e:
            print(f"Error in RAG pipeline: {e}")
            # Try to retrieve documents even if LLM fails
            try:
                docs = self.retriever.get_relevant_documents(question)
                return {
                    "question": question,
                    "answer": f"Error generating response: {str(e)}",
                    "source_documents": docs
                }
            except Exception as retrieval_error:
                return {
                    "question": question,
                    "answer": f"Error in RAG pipeline: {str(e)}, Retrieval error: {str(retrieval_error)}",
                    "source_documents": []
                }
    
    def retrieve_documents(self, question: str, k: int = config.DEFAULT_TOP_K) -> List[Document]:
        search_kwargs = {"k": k} if k is not None else None
        return self.retriever.get_relevant_documents(question, search_kwargs=search_kwargs)
    
    def get_retriever(self) -> BaseRetriever:
        return self.retriever

In [ ]:
rag_system = SimplifiedRAG()

def generate(prompt):
    result = rag_system.ask(prompt)
    # For API responses, return just the answer text
    return result["answer"]

In [ ]:
# Step 1: Download ngrok v3
!wget -q -O ngrok.zip https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.zip
!unzip -o ngrok.zip
!mv ngrok /usr/local/bin/ngrok
!chmod +x /usr/local/bin/ngrok

# Step 2: Add your ngrok authtoken (from https://dashboard.ngrok.com/get-started/setup)
!ngrok config add-authtoken authtoken

In [ ]:
from flask import Flask, request, jsonify
import threading

app = Flask(__name__)

@app.route("/")
def home():
    return "Phi-2 Flask API đang chạy!"

@app.route("/generate", methods=["POST"])
def generate_text():
    data = request.json
    prompt = data.get("prompt", "")
    result = generate(prompt)
    return jsonify({"response": result})
@app.route("/generate", methods=["GET"])
def generate_text_():
    prompt = request.args.get("prompt")
    result = generate(prompt)
    return jsonify({"response": result})
def run_flask():
    app.run(port=5001)

thread = threading.Thread(target=run_flask)
thread.start()

!ngrok http 5001